# Final evaluation: Motion Light detection (TON_IoT)
This notebook **only evaluates**. It does not train anything. It reads the three files exported by the training notebook (`Motion_Light_IoT_Detection_final.ipynb`):

- `export/motion_light_predictions.csv`
- `export/motion_light_events.csv`
- `export/motion_light_config.json`

## How to use it
1. Run the training notebook first, so these files exist (or upload them when this notebook asks).
2. Run all the cells here. The results are saved in `results/evaluation_*.csv`.


In [ ]:
import os, json, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.metrics import (precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", rc={"axes.grid.axis": "y"})
plt.rcParams["figure.dpi"] = 100
SEED = 42
SOURCE = "motion_light"
BLOCK = 100                                   # windows per block (same value as the training notebook)
EXPORT_DIR = os.environ.get("EXPORT_DIR", "export")
NEEDED = [f"{SOURCE}_predictions.csv", f"{SOURCE}_events.csv", f"{SOURCE}_config.json"]

# if the files are not here (e.g. a fresh Colab session), ask for them
if not all(os.path.exists(f"{EXPORT_DIR}/{f}") for f in NEEDED):
    from google.colab import files
    os.makedirs(EXPORT_DIR, exist_ok=True)
    for name, data in files.upload().items():
        open(f"{EXPORT_DIR}/{name}", "wb").write(data)
os.makedirs("results", exist_ok=True)
print("files found:", [f for f in NEEDED if os.path.exists(f"{EXPORT_DIR}/{f}")])

<style>
.wbox, .wbox * { color:#111111 !important; }
.wbox { background-color:#ffffff !important; border:6px solid #ffffff; border-radius:12px; padding:18px 26px; font-family:Arial, sans-serif; line-height:1.7; box-shadow:0 0 0 1px #cccccc; }
.wbox h2, .wbox h3 { margin-top:10px; }
</style>
<div class="wbox">
<h2>Final evaluation</h2>
<ul>
  <li>This section measures the <b>final model</b> exactly as it will be used by the next steps: it reads <code>export/motion_light_predictions.csv</code>, <code>_events.csv</code> and <code>_config.json</code>, not the variables in memory.</li>
  <li>The headline numbers are the <b>test</b> ones. Train and val are shown only to check overfitting (a big gap between train and test is a warning).</li>
</ul>
<h3>How to read the metrics</h3>
<ul>
  <li><b>Recall:</b> from all real attacks, how many did we catch? (missing an attack is the expensive mistake)</li>
  <li><b>Precision:</b> from everything we flagged as Attack, how much was really an attack? (low precision = many false alarms)</li>
  <li><b>F1:</b> one number that balances recall and precision.</li>
  <li><b>FPR (false-alarm rate):</b> from the normal windows, how many did we wrongly flag. <b>Specificity</b> = 1 - FPR.</li>
  <li><b>ROC-AUC / PR-AUC:</b> quality of the probability itself, independent of the threshold. PR-AUC matters more here because attacks are the minority.</li>
  <li><b>MCC / balanced accuracy:</b> honest summaries for imbalanced data (plain accuracy can look good by always saying Normal).</li>
  <li><b>Brier / ECE:</b> how close the probabilities are to reality (calibration). Lower is better.</li>
</ul>
<h3>What each part checks</h3>
<ul>
  <li><b>Metrics table:</b> overall quality on train / val / test.</li>
  <li><b>Confusion matrix and per-type detection:</b> where the mistakes are, and which attack types are missed.</li>
  <li><b>Multi-class report:</b> can the model name the attack type? Types with no window in the test set are listed and cannot be scored.</li>
  <li><b>Reliability curve:</b> can we trust <code>p_attack</code> as a real probability in the risk step?</li>
  <li><b>Anomaly score:</b> does it separate attacks, and is it really different from the attack probability?</li>
  <li><b>Block bootstrap:</b> a 95% confidence interval, to see how much the test result could change with different blocks.</li>
</ul>
</div>

# 1) Metrics on train / val / test

In [ ]:
# Final evaluation. Everything below is computed ONLY from the exported files, i.e. exactly what the aggregation step will read.
from sklearn.metrics import accuracy_score, balanced_accuracy_score, matthews_corrcoef, average_precision_score, brier_score_loss, classification_report
from sklearn.calibration import calibration_curve

pred_df = pd.read_csv(f"{EXPORT_DIR}/{SOURCE}_predictions.csv")
ev_df = pd.read_csv(f"{EXPORT_DIR}/{SOURCE}_events.csv")
with open(f"{EXPORT_DIR}/{SOURCE}_config.json") as f:
    cfg_ev = json.load(f)
TH = cfg_ev["threshold"]                                  # the threshold chosen on validation
pred_df["y_pred"] = (pred_df["p_attack"] >= TH).astype(int)

# sanity checks on the exported files
PRED_COLS = ["source", "chunk", "win", "split", "p_attack", "y_true", "true_type", "pred_type", "n_records"]
assert list(pred_df.columns)[:len(PRED_COLS)] == PRED_COLS, "predictions columns / order are wrong"
assert set(pred_df["split"]) <= {"train", "val", "test"}, "unexpected split values (purged rows must not be exported)"
assert len(pred_df) == len(ev_df), "events and predictions must have the same rows"
assert ((pred_df["p_attack"].values >= TH) == (ev_df["prediction"].values == "Attack")).all(), "events and predictions disagree"
print("export files are consistent")

def binary_metrics(d):
    y_, p_, yh = d["y_true"].values, d["p_attack"].values, d["y_pred"].values
    tn, fp, fn, tp = confusion_matrix(y_, yh, labels=[0, 1]).ravel()
    return dict(windows=len(d), attack_rate=y_.mean(),
                accuracy=accuracy_score(y_, yh), balanced_acc=balanced_accuracy_score(y_, yh),
                precision=precision_score(y_, yh, zero_division=0), recall=recall_score(y_, yh),
                specificity=tn / (tn + fp), FPR=fp / (fp + tn), FNR=fn / (fn + tp),
                f1=f1_score(y_, yh), mcc=matthews_corrcoef(y_, yh),
                roc_auc=roc_auc_score(y_, p_), pr_auc=average_precision_score(y_, p_), brier=brier_score_loss(y_, p_))

eval_table = pd.DataFrame({s: binary_metrics(pred_df[pred_df["split"] == s]) for s in ["train", "val", "test"]}).T.round(3)
print(f"Model: {cfg_ev['binary_model']} | threshold = {TH:.3f}")
eval_table

# 2) Confusion matrix (test)

In [ ]:
# Confusion matrix on the TEST set (counts and percentages of each real class)
te_ = pred_df[pred_df["split"] == "test"].copy()
cm_ = confusion_matrix(te_["y_true"], te_["y_pred"], labels=[0, 1])
cm_pct = cm_ / cm_.sum(axis=1, keepdims=True)
labels_ = [f"{c}\n({p:.1%})" for c, p in zip(cm_.ravel(), cm_pct.ravel())]
plt.figure(figsize=(5.5, 4.5))
sns.heatmap(cm_pct, annot=np.array(labels_).reshape(2, 2), fmt="", cmap="Blues", cbar=False,
            xticklabels=["Normal", "Attack"], yticklabels=["Normal", "Attack"])
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Test confusion matrix (row %)")
plt.tight_layout(); plt.show()

# 3) Detection rate per attack type

In [ ]:
# Which attack types are caught? Share of test windows predicted as Attack, per real type.
# For the attack types this is the detection rate (recall). For "normal" it is the false-alarm rate.
by_type = te_.groupby("true_type").agg(windows=("y_true", "size"), predicted_attack=("y_pred", "sum"))
by_type["share_predicted_attack"] = (by_type["predicted_attack"] / by_type["windows"]).round(3)
by_type = by_type.sort_values("windows", ascending=False)

fig, ax = plt.subplots(figsize=(9, 4))
colors = ["#4c72b0" if t == "normal" else "#c44e52" for t in by_type.index]
ax.bar(by_type.index, by_type["share_predicted_attack"], color=colors)
ax.set_ylim(0, 1.05); ax.set_ylabel("Share predicted as Attack")
ax.set_title("Detection rate per attack type (blue = normal = false alarm rate)")
for i, (w, v) in enumerate(zip(by_type["windows"], by_type["share_predicted_attack"])):
    ax.text(i, v + 0.02, f"n={w}", ha="center", fontsize=8)
plt.xticks(rotation=30); plt.tight_layout(); plt.show()
by_type

# 4) Multi-class report (attack type)

In [ ]:
# Multi-class report (attack type) on the TEST set: only the types that really exist in the test set are scored
present_types = sorted(te_["true_type"].unique())
print(classification_report(te_["true_type"], te_["pred_type"], labels=present_types, zero_division=0, digits=3))
missing = sorted(set(pred_df["true_type"]) - set(present_types))
print("Types with NO test window (cannot be evaluated):", missing)

# 5) Calibration of the attack probability

In [ ]:
# Calibration: when the model says 0.8, is it really an attack about 80% of the time?
frac_pos, mean_pred = calibration_curve(te_["y_true"], te_["p_attack"], n_bins=10, strategy="quantile")
ece = np.mean(np.abs(frac_pos - mean_pred))                      # average gap between the two (lower is better)
plt.figure(figsize=(5.5, 5))
plt.plot([0, 1], [0, 1], "k--", lw=0.8, label="perfect calibration")
plt.plot(mean_pred, frac_pos, marker="o", label=f"model (ECE ~ {ece:.3f})")
plt.xlabel("Predicted attack probability"); plt.ylabel("Real attack fraction")
plt.title("Reliability curve (test)"); plt.legend(); plt.tight_layout(); plt.show()

# 6) Anomaly score

In [ ]:
# Anomaly score: does it separate Attack from Normal, and is it different from the attack probability?
ev_te = ev_df[ev_df["split"] == "test"]
print(f"Anomaly method: {cfg_ev['anomaly_method']}")
print(f"AUC = {roc_auc_score(ev_te['true_label'], ev_te['anomaly_score']):.3f} | "
      f"Average precision = {average_precision_score(ev_te['true_label'], ev_te['anomaly_score']):.3f} "
      f"(random score would give {ev_te['true_label'].mean():.3f})")
print(f"Spearman correlation with attack_prob = {ev_te['anomaly_score'].corr(ev_te['attack_prob'], method='spearman'):.3f}")

plt.figure(figsize=(8, 3.5))
for lab, col, nm in ((0, "#4c72b0", "Normal"), (1, "#c44e52", "Attack")):
    plt.hist(ev_te.loc[ev_te["true_label"] == lab, "anomaly_score"], bins=30, alpha=0.6, density=True, color=col, label=nm)
plt.xlabel("anomaly_score (0 = like normal, 1 = very unusual)"); plt.legend(); plt.title("Anomaly score on the test set")
plt.tight_layout(); plt.show()

# 7) Stability: block bootstrap confidence interval

In [ ]:
# How stable is the test result? Block bootstrap: resample whole 100-window blocks (neighbouring windows are not independent)
te_["block"] = te_["win"] // BLOCK
cnt = te_.groupby("block").apply(lambda d: pd.Series(dict(
    tp=int(((d.y_true == 1) & (d.y_pred == 1)).sum()),
    fp=int(((d.y_true == 0) & (d.y_pred == 1)).sum()),
    fn=int(((d.y_true == 1) & (d.y_pred == 0)).sum())))).values
rng = np.random.default_rng(SEED); n_b = len(cnt); rows = []
for _ in range(1000):
    tp_, fp_, fn_ = cnt[rng.integers(0, n_b, n_b)].sum(axis=0)
    pr_, rc_ = tp_ / max(tp_ + fp_, 1), tp_ / max(tp_ + fn_, 1)
    rows.append((pr_, rc_, 2 * pr_ * rc_ / max(pr_ + rc_, 1e-12)))
ci = pd.DataFrame(rows, columns=["precision", "recall", "f1"]).quantile([0.025, 0.5, 0.975]).T.round(3)
ci.columns = ["2.5%", "median", "97.5%"]
print(f"Test blocks: {n_b}  (95% confidence interval from 1000 resamples)")
ci

# 8) Final check and save

In [ ]:
# Final check + save
assert abs(eval_table.loc["test", "f1"] - cfg_ev["test_f1"]) < 1e-3, "evaluation F1 does not match the config test_f1"
final_eval = pd.concat({"binary_metrics": eval_table.T, "attack_type_detection": by_type.T}, axis=0)
eval_table.to_csv("results/evaluation_binary.csv"); by_type.to_csv("results/evaluation_by_attack_type.csv"); ci.to_csv("results/evaluation_confidence_intervals.csv")
print("Saved: results/evaluation_binary.csv, evaluation_by_attack_type.csv, evaluation_confidence_intervals.csv")
print(f"TEST  F1={eval_table.loc['test','f1']:.3f} | recall={eval_table.loc['test','recall']:.3f} | precision={eval_table.loc['test','precision']:.3f} | false-alarm rate={eval_table.loc['test','FPR']:.3f}")